In [ ]:
!pip install -U -q langchain-community
!pip install -U -q langchain_core
!pip install -U -q gigachain-community
!pip install -U -q pypdf
!pip install -U -q chromadb
!pip install -U -q tiktoken
!pip install -U -q langchain_experimental
!pip install -U -q rank_bm25
!pip install -U -q rouge
!pip install -U -q transformers

In [ ]:
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/main/dataset/clean/requirements.csv
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/main/dataset/clean/risk1.csv
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/main/dataset/clean/risk2.csv
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/luzanin/dataset/data_for_pipeline/queries1.csv
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/luzanin/dataset/data_for_pipeline/queries2.csv
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/luzanin/dataset/data_for_pipeline/queries3.csv
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/luzanin/dataset/data_for_pipeline/end_to_end.csv
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/luzanin/retrieval_modules.py
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/luzanin/retriever_validation.py
!wget -q https://raw.githubusercontent.com/artyomrabosh/rag_for_sber/refs/heads/aandreev/validation_generation.py

In [ ]:
import pandas as pd
import re
import pickle
import torch

from tqdm import tqdm

from scipy.spatial.distance import cosine

import numpy as np

from IPython.display import clear_output

from langchain.document_loaders import PyPDFLoader
import langchain_core
from langchain_core.documents.base import Document
from langchain.vectorstores import Chroma
import langchain
from langchain.chat_models import gigachat
from langchain.schema import HumanMessage, SystemMessage
from langchain.chat_models.gigachat import GigaChat

from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from sentence_transformers import SentenceTransformer

from rank_bm25 import BM25Okapi

import nltk
from nltk import WordPunctTokenizer
from nltk.corpus import stopwords

nltk.download('punkt_tab')
nltk.download('stopwords')
russian_stopwords = stopwords.words("russian")

import chromadb
chroma_client = chromadb.Client()

from google.colab import userdata
API_TOKEN = userdata.get('GIGACHAT')


from retrieval_modules import *
from retriever_validation import *
from validation_generation import *

from langchain.text_splitter import (RecursiveCharacterTextSplitter,
                                    SentenceTransformersTokenTextSplitter,
                                    TokenTextSplitter,
                                    NLTKTextSplitter,
                                    SpacyTextSplitter
                                    )


device = 'cuda' if torch.cuda.is_available() else 'cpu'


from splitters import all_splitters

### Читаем документы и вопросы

##### Загружаем документы

In [ ]:
document_1 = pd.read_csv('risk1.csv')
document_2 = pd.read_csv('risk2.csv')
document_3 = pd.read_csv('requirements.csv')

In [ ]:
docs_1 = []
docs_2 = []
docs_3 = []
for row in document_1.iterrows():
    docs_1.append(Document(row[1].loc['content']))
    docs_1[-1].metadata = {'header_1': row[1].loc['Header_1'],
                           'header_2': row[1].loc['Header_2'],
                           'header_3': row[1].loc['Header_3']}

for row in document_2.iterrows():
    docs_2.append(Document(row[1].loc['content']))
    docs_2[-1].metadata = {'header_1': row[1].loc['Header_1'],
                           'header_2': row[1].loc['Header_2'],
                           'header_3': row[1].loc['Header_3']}

for row in document_3.iterrows():
    docs_3.append(Document(row[1].loc['content']))
    docs_3[-1].metadata = {'header_1': row[1].loc['Header_1'],
                           'header_2': row[1].loc['Header_2'],
                           'header_3': row[1].loc['Header_3']}


In [ ]:
whole_doc = docs_1 + docs_2 + docs_3

##### Загружаем датасет с вопросами

In [ ]:
queries_1 = pd.read_csv('queries1.csv')
queries_2 = pd.read_csv('queries2.csv')
queries_3 = pd.read_csv('queries3.csv')
end_to_end = pd.read_csv('end_to_end.csv')

## Валидация ретривера

In [ ]:
def count_all_results(splitters, embedder):
    results = {}
    for i, splitter in tqdm(enumerate(splitters)):

        splitted_docs = splitter.split_documents(whole_doc)

        vectordb = Chroma.from_documents(
            documents=splitted_docs,
            embedding=embedder,
            persist_directory=f'docs/{i}/'
        )

        vectordb.persist()
        params = [{'db' : vectordb, 'strategy' : 'mmr', 'fusion_alpha' : 1.},
                {'db' : vectordb, 'strategy' : 'ss', 'fusion_alpha' : 1.},
                {'db' : vectordb, 'strategy' : 'ss', 'fusion_alpha' : 0.8},
                {'db' : vectordb, 'strategy' : 'ss', 'fusion_alpha' : 0.6},
                {'db' : vectordb, 'strategy' : 'ss', 'fusion_alpha' : 0.4}]



        for param in params:
            name = splitter.name + param['strategy'] + str(param['fusion_alpha'])
            retriever = Retriever(**param)

            results[name] = count_metrics(retriever, 5, pd.concat([queries_1, queries_2, queries_3]), embedder)
    return results

### Валидируем эмбеддер, сплиттер, стретегию, fusion

#### E5 large embedder

In [ ]:
embedder_e5 = SentenceTransformer("intfloat/multilingual-e5-large").to(device)
embedder_e5 = Embedder_wrapper(embedder_e5)

e5_large_results = count_all_results(all_splitters, embedder_e5)

with open('./results_e5.pkl', 'wb') as f:
    pickle.dump(e5_large_results, f)

#### E5 large instruct embedder

In [ ]:
embedder_e5_instruct = SentenceTransformer('intfloat/multilingual-e5-large-instruct').to(device)
embedder_e5_instruct = Embedder_wrapper_e5_instruct(embedder_e5_instruct)

e5_large_instruct_results = count_all_results(all_splitters, embedder_e5_instruct)

with open('./results_e5_instruct.pkl', 'wb') as f:
    pickle.dump(e5_large_instruct_results, f)

#### Nomic embedder

In [ ]:
embedder_nomic = SentenceTransformer("nomic-ai/nomic-embed-text-v1", trust_remote_code=True).to(device)
embedder_nomic = Embedder_wrapper_nomic(embedder_nomic)

nomic_results = count_all_results(all_splitters, embedder_nomic)

with open('./results_nomic.pkl', 'wb') as f:
    pickle.dump(nomic_results, f)

#### Сравнение результатов

In [ ]:
with open('./results_e5.pkl', 'rb') as f:
    e5_large_results = pickle.load(f)

with open('./results_e5_instruct.pkl', 'rb') as f:
    e5_large_instruct_results = pickle.load(f)

with open('./results_nomic.pkl', 'rb') as f:
    nomic_results = pickle.load(f)

In [ ]:
get_best_result(e5_large_results)

In [ ]:
get_best_result(e5_large_instruct_results)

In [ ]:
get_best_result(nomic_results)

#### Вывод


Лучший результат достигается при таких параметрах: \
embedder = E5_large_instruct \
splitter = recursive_text_splitter \
chunk_size = 2084 \
chunk_overlap = 0 \
strategy = similarity search \
fusion = 0.8